# E3: giữ hay xóa dòng trùng

**Câu hỏi:** quyết định xóa dòng trùng hoàn toàn trong bước làm sạch ảnh hưởng thế nào đến mô hình và đến con số báo cáo?

**Thiết kế**
- Hai bản dữ liệu: *dedup* (xóa trùng, bản chính, 86.617 dòng) và *keepdup* (giữ dòng trùng, tạo bằng `--keep-duplicates`).
- Cả hai chia theo **đúng ranh giới thời gian** như notebook 02, và cùng chạy hai mô hình tham chiếu mặc định (Logistic Regression, Random Forest), không tinh chỉnh.
- Chấm theo hai cách, vì chúng trả lời hai câu hỏi khác nhau:
  1. **Trên tập kiểm định riêng của từng bản:** con số mà mỗi cách làm sạch sẽ "tự báo cáo". Hai tập kiểm định có số dòng và tỷ lệ hủy khác nhau nên PR-AUC không so sánh thẳng được (mốc của PR-AUC chính là tỷ lệ hủy).
  2. **Trên cùng một tập kiểm định sạch (bản dedup):** so sánh công bằng xem *huấn luyện* trên dữ liệu có dòng trùng làm mô hình tốt lên hay kém đi.
- Chỉ dùng `train` và `val`; không mở `test`.

**Lưu ý:** dòng trùng giống hệt nhau cả ngày đến, nên chúng luôn rơi vào cùng một tập khi chia theo thời gian; bản sao không bao giờ nằm vắt qua train và val. Tác hại của việc giữ trùng vì vậy chỉ là (a) các đơn bị lặp có trọng số lớn hơn khi huấn luyện, (b) các đơn bị lặp được tính nhiều lần khi chấm điểm.

In [ ]:
import sys, subprocess; sys.path.insert(0, "..")

import pandas as pd

from src.data import REPO_ROOT, TARGET, load_split
from src.split import time_split
from src.reference import make_reference, REFERENCE
from src.evaluate import evaluate, log_result
from src.compare import paired_bootstrap

AUTHOR = "tuan"
SHOW = ["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy", "n", "positive_rate"]

RAW = REPO_ROOT / "data" / "raw" / "hotel_bookings.csv"
MAIN = REPO_ROOT / "data" / "interim" / "hotel_bookings_clean.csv"
KEEPDUP = REPO_ROOT / "data" / "interim" / "hotel_bookings_keepdup.csv"
SCRIPT = REPO_ROOT / "src" / "clean_hotel_bookings.py"

## Tạo bản giữ dòng trùng

Chạy lại script làm sạch với công tắc `--keep-duplicates`; mọi bước khác giữ nguyên.

In [ ]:
res = subprocess.run([sys.executable, str(SCRIPT), str(RAW), str(KEEPDUP), "--keep-duplicates"],
                     capture_output=True, text=True, check=True)
print(res.stdout)

main = pd.read_csv(MAIN)
dup = pd.read_csv(KEEPDUP)
assert list(main.columns) == list(dup.columns)
print(f"dedup: {len(main):,} dòng | keepdup: {len(dup):,} dòng | khác nhau: {len(dup) - len(main):,} dòng trùng")

## Chia theo thời gian và so sánh hai bản

In [ ]:
parts_main = time_split(main)
parts_dup = time_split(dup)

# kiểm tra: chia bản dedup ở đây phải khớp với file đã chia ở notebook 02
for s in ("train", "val"):
    assert len(parts_main[s]) == len(load_split(s)), f"{s} không khớp notebook 02"

rows = {}
for s in ("train", "val"):
    extra = len(parts_dup[s]) - len(parts_main[s])
    extra_cancel = (parts_dup[s][TARGET].sum() - parts_main[s][TARGET].sum()) / extra
    rows[s] = {"dòng dedup": len(parts_main[s]), "dòng keepdup": len(parts_dup[s]),
               "dòng trùng thêm": extra, "tỷ lệ hủy dedup": parts_main[s][TARGET].mean(),
               "tỷ lệ hủy keepdup": parts_dup[s][TARGET].mean(),
               "tỷ lệ hủy của riêng các dòng trùng": extra_cancel}
pd.DataFrame(rows).T.round(3)

## Huấn luyện và chấm điểm

Với mỗi mô hình tham chiếu: huấn luyện trên *train* của từng bản, rồi chấm trên *val*. Riêng mô hình huấn luyện từ keepdup được chấm hai lần (val riêng và val sạch chung).

In [ ]:
def xy(df):
    return df.drop(columns=[TARGET]), df[TARGET]

Xtr_m, ytr_m = xy(parts_main["train"]); Xv_m, yv_m = xy(parts_main["val"])
Xtr_d, ytr_d = xy(parts_dup["train"]);  Xv_d, yv_d = xy(parts_dup["val"])

rows, probas = [], {}
for name in REFERENCE:
    m_dedup = make_reference(name).fit(Xtr_m, ytr_m)
    m_dup = make_reference(name).fit(Xtr_d, ytr_d)

    probas[(name, "dedup")] = m_dedup.predict_proba(Xv_m)[:, 1]
    probas[(name, "keepdup_common")] = m_dup.predict_proba(Xv_m)[:, 1]
    p_own = m_dup.predict_proba(Xv_d)[:, 1]

    for note, y, p in [("dedup", yv_m, probas[(name, "dedup")]),
                       ("keepdup_own_val", yv_d, p_own),
                       ("keepdup_common_val", yv_m, probas[(name, "keepdup_common")])]:
        m = evaluate(y, p)
        log_result(AUTHOR, "E3", name, "val", {**m, "params": "default"}, note=note)
        rows.append({"mô hình": name, "cách làm": note, **m})

e3 = pd.DataFrame(rows).set_index(["mô hình", "cách làm"])[SHOW].round(3)
e3

**Cách đọc bảng**
- `dedup`: huấn luyện và chấm trên bản đã xóa trùng (bản chính của nhóm).
- `keepdup_own_val`: huấn luyện trên bản giữ trùng, chấm trên tập kiểm định *có* dòng trùng. Đây là con số mà quyết định "giữ trùng" sẽ tự báo cáo. Để ý `n` và `positive_rate` khác bản dedup.
- `keepdup_common_val`: huấn luyện trên bản giữ trùng, chấm trên **cùng** tập kiểm định với `dedup`. Chỉ dòng này so sánh công bằng với `dedup`.

## Chênh lệch có lớn hơn nhiễu không?

So sánh `keepdup_common_val` với `dedup` trên cùng các dòng kiểm định bằng bootstrap ghép cặp. Chênh lệch dương nghĩa là huấn luyện trên bản giữ trùng *tốt hơn*.

In [ ]:
rows = []
for name in REFERENCE:
    for metric in ("pr_auc", "roc_auc"):
        r = paired_bootstrap(yv_m, probas[(name, "dedup")], probas[(name, "keepdup_common")], metric)
        rows.append({"mô hình": name, "chỉ số": metric, **r})
pd.DataFrame(rows).set_index(["mô hình", "chỉ số"]).round(4)

Bootstrap này chỉ đo nhiễu do chọn mẫu kiểm định, không đo nhiễu do huấn luyện (Random Forest đổi seed sẽ lệch thêm một chút). Nếu khoảng tin cậy chứa 0 thì kết luận đúng là "chưa thấy khác biệt rõ", không phải "hai cách bằng nhau".

## Kết luận E3 (điền sau khi chạy)

- Quyết định giữ hay xóa trùng thay đổi **con số báo cáo** bao nhiêu (so `dedup` với `keepdup_own_val`)? Phần lớn là do tỷ lệ hủy và quy mô tập kiểm định khác nhau hay do chất lượng mô hình?
- Xét trên **cùng tập kiểm định**, huấn luyện có dòng trùng làm mô hình tốt hơn, kém hơn hay không khác, và khác bao nhiêu so với nhiễu?
- Cleaning log ghi dòng trùng hủy ít hơn (hoặc nhiều hơn) mức chung bao nhiêu; chiều thay đổi tỷ lệ hủy tổng thể khi xóa trùng có ảnh hưởng tới câu hỏi chi phí ở E8 không?
- Lựa chọn của nhóm là xóa trùng. Bằng chứng ở đây ủng hộ hay không ủng hộ lựa chọn đó? Nhớ nêu giới hạn: không có mã đặt phòng nên không biết dòng trùng là bản ghi lặp hay hai đơn thật giống nhau.